# FoxTrack 02: Baseline early-detection model

This notebook trains a baseline detector on benign traffic only, then measures how many packets pass between the start of each attack and the first alert.

**Before you run it (right panel of the Kaggle editor):**

1. In notebook 01, select **Save Version > Save & Run All (Commit)** and wait until it completes. This keeps the Parquet output.
2. Here, select **Input > Add Input > Notebook Output** and add notebook 01.
3. **Settings > Internet:** set to on.
4. **Private repository only:** attach the `GITHUB_TOKEN` secret to this notebook.

**Method.** Each capture has a long benign period, then one attack start. For each capture:

| Range of the benign period | Use |
|---|---|
| First 70% | Fit a scaler and a PCA (Spark MLlib) |
| Next 15% | Set the alert threshold for a target false positive rate |
| Last 15%, and all packets after it | Test: includes the attack start |

The score of a packet is its PCA reconstruction error. Alert rules: `threshold` (one score more than the threshold), `3 of 5` and `10 of 20` (k of the last n scores), and `mean of 100` (the moving mean of the last 100 scores is more than its threshold).

In [1]:
# Settings.
REPO = "DiscM/DATA-226-FoxTrack"
BRANCH = "main"
OUT_DIR = "/kaggle/working/baseline"
K_COMPONENTS = 20                      # PCA dimensions
TARGET_FPRS = (1e-2, 1e-3, 1e-4)       # target false positive rates for the threshold
ALERT_RULES = ((1, 1), (3, 5), (10, 20))  # (k, n): k of the last n scores
SMOOTH_WINDOW = 100                    # packets in the moving mean of the score

In [2]:
import shutil, subprocess, sys
if shutil.which("java") is None:
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless", shell=True, check=True)
!pip install -q pyspark mlflow

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None  # a public repository does not need a token
auth = f"{token}@" if token else ""
# A new clone on each run, so the code is always the latest version.
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
done = subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
     f"https://{auth}github.com/{REPO}.git", "/kaggle/working/repo"],
    capture_output=True, text=True,
)
if done.returncode:
    raise RuntimeError("git clone failed. Examine REPO, BRANCH, and the GITHUB_TOKEN secret.")
sys.path.insert(0, "/kaggle/working/repo/src")
from foxtrack import anomaly, etl, split
print("Code version:", subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

Code version: c6ff91d Get the demo packets from the dataset, not from the repository


In [3]:
# Find the Parquet output of notebook 01.
from pathlib import Path
found = sorted(Path("/kaggle/input").rglob("summary.json")) + sorted(Path("/kaggle/working").glob("*/summary.json"))
if not found:
    raise FileNotFoundError("summary.json not found. Add the output of notebook 01 in Input > Add Input > Notebook Output.")
PARQUET_ROOT = found[0].parent
print("Parquet data:", PARQUET_ROOT)

Parquet data: /kaggle/input/notebooks/<kaggle-user>/notebookabc04dfe5a/kitsune_parquet


## 1. Train and evaluate each capture

In [4]:
%%time
import json, os
import mlflow
import numpy as np
import pandas as pd

spark = etl.build_spark(driver_memory="20g")
mlflow.set_tracking_uri("sqlite:////kaggle/working/mlflow.db")
mlflow.set_experiment("baseline_pca_onset")
for folder in ("scores", "models", "demo"):
    os.makedirs(f"{OUT_DIR}/{folder}", exist_ok=True)

all_results = []
for row in split.load_summary(PARQUET_ROOT):
    s = split.onset_split(row)
    capture = spark.read.parquet(f"{PARQUET_ROOT}/features/attack={s.attack}")
    results, test, model = anomaly.run_capture(capture, s, K_COMPONENTS, TARGET_FPRS, ALERT_RULES, (1, SMOOTH_WINDOW))
    all_results.append(results)
    test.to_parquet(f"{OUT_DIR}/scores/{s.attack}.parquet", index=False)

    # The trained model as plain numbers, and packets near the attack start, for the service and the dashboard.
    model_path = f"{OUT_DIR}/models/{s.attack}.json"
    with open(model_path, "w") as fh:
        json.dump(anomaly.model_artifact(model, results, s), fh)
    np.savez_compressed(f"{OUT_DIR}/demo/{s.attack}.npz", **anomaly.demo_window(capture, s))

    with mlflow.start_run(run_name=s.attack):
        mlflow.log_artifact(model_path, artifact_path="model")
        mlflow.log_params({"attack": s.attack, "model": "scaler+pca", "k_components": K_COMPONENTS,
                           "train_end": s.train_end, "calib_end": s.calib_end, "onset": s.onset})
        for r in results.itertuples():
            tag = f"fpr{r.target_fpr:g}_{r.alert_rule.replace(' ', '_')}"
            mlflow.log_metric(f"false_alarms_per_100k_{tag}", r.false_alarms_per_100k)
            mlflow.log_metric(f"packet_recall_{tag}", r.packet_recall)
            if r.detected:
                mlflow.log_metric(f"delay_packets_{tag}", r.delay_packets)
    print(f"{s.attack}: done")

results = pd.concat(all_results, ignore_index=True)
results.to_csv(f"{OUT_DIR}/results.csv", index=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/22 06:49:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026/11/22 06:49:44 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/22 06:49:44 INFO mlflow.store.db.utils: Updating database tables
2026/11/22 06:49:47 INFO mlflow.tracking.fluent: Experiment with name 'baseline_pca_onset' does not exist. Creating a new experiment.
26/11/22 06:49:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


arp_mitm: done
active_wiretap: done
fuzzing: done
mirai_botnet: done
os_scan: done
ssdp_flood: done
ssl_renegotiation: done
syn_dos: done
video_injection: done
CPU times: user 1min 13s, sys: 6.28 s, total: 1min 19s
Wall time: 10min 19s


## 2. Results

**Detection delay in packets** (empty = the attack was not detected). Rows are attacks. Columns are the target false positive rate and the alert rule.

In [5]:
results.pivot(index="attack", columns=["target_fpr", "alert_rule"], values="delay_packets")

target_fpr           0.0100                      0.0010                    \
alert_rule        threshold   3 of 5 10 of 20 threshold   3 of 5 10 of 20   
attack                                                                      
active_wiretap          0.0      0.0      0.0    1145.0   3838.0   3845.0   
arp_mitm                0.0      0.0      0.0      46.0  16626.0  16633.0   
fuzzing              1028.0   5006.0   5338.0    4310.0   5006.0   5338.0   
mirai_botnet           13.0     15.0     48.0      41.0     43.0    259.0   
os_scan                 0.0     31.0   1536.0       0.0     31.0   1536.0   
ssdp_flood            424.0    776.0    826.0     424.0   4052.0   4060.0   
ssl_renegotiation      25.0     27.0    624.0    2841.0   4229.0  24950.0   
syn_dos               218.0   9558.0   9996.0     218.0  27231.0  27238.0   
video_injection       256.0  13093.0  28806.0    6743.0  26465.0  28806.0   

target_fpr           0.0001                        0.0100      0.0010  \
alert_rule        threshold   3 of 5 10 of 20 mean of 100 mean of 100   
attack                                                                  
active_wiretap       1145.0   3907.0   3914.0         0.0         0.0   
arp_mitm             2290.0  24452.0  60262.0         0.0      8591.0   
fuzzing             17411.0  46500.0  46507.0      5070.0    500304.0   
mirai_botnet          150.0    613.0   1115.0        41.0        42.0   
os_scan                 0.0     31.0   1536.0         0.0        29.0   
ssdp_flood            424.0   4057.0   4072.0       778.0      4059.0   
ssl_renegotiation    3037.0   4229.0  59278.0      3037.0      4035.0   
syn_dos             19520.0  27341.0  27375.0     12346.0     12424.0   
video_injection      6744.0  26466.0      NaN     28615.0     29156.0   

target_fpr             0.0001  
alert_rule        mean of 100  
attack                         
active_wiretap           34.0  
arp_mitm              16629.0  
fuzzing              500331.0  
mirai_botnet             42.0  
os_scan                  30.0  
ssdp_flood             4077.0  
ssl_renegotiation      4035.0  
syn_dos               27261.0  
video_injection           NaN

**Credible delay in packets.** This table shows a delay only if two checks pass:

- The alert rate on malicious packets is at least 5 times the alert rate on benign packets before the attack.
- A detector with no skill and the same false alarm rate would give an alert that quickly in at most 5% of cases.

An empty cell shows that false alarms can explain the alert. Use this table for the report, not the table of all delays.

In [6]:
results.assign(credible=results["delay_packets"].where(results["delay_credible"])).pivot(index="attack", columns=["target_fpr", "alert_rule"], values="credible")

target_fpr           0.0100                    0.0010                   \
alert_rule        threshold 3 of 5 10 of 20 threshold  3 of 5 10 of 20   
attack                                                                   
active_wiretap          NaN    NaN      NaN       NaN  3838.0   3845.0   
arp_mitm                NaN    NaN      NaN       NaN     NaN      NaN   
fuzzing                 NaN    NaN      NaN       NaN     NaN      NaN   
mirai_botnet            NaN   15.0     48.0      41.0    43.0    259.0   
os_scan                 0.0   31.0      NaN       0.0    31.0   1536.0   
ssdp_flood              NaN    NaN      NaN       NaN     NaN      NaN   
ssl_renegotiation      25.0   27.0      NaN       NaN     NaN      NaN   
syn_dos                 NaN    NaN      NaN       NaN     NaN      NaN   
video_injection         NaN    NaN      NaN       NaN     NaN      NaN   

target_fpr           0.0001                       0.0100      0.0010  \
alert_rule        threshold  3 of 5 10 of 20 mean of 100 mean of 100   
attack                                                                 
active_wiretap          NaN  3907.0   3914.0         NaN         NaN   
arp_mitm                NaN     NaN      NaN         NaN         NaN   
fuzzing                 NaN     NaN      NaN         NaN         NaN   
mirai_botnet          150.0   613.0      NaN        41.0        42.0   
os_scan                 0.0    31.0   1536.0         0.0        29.0   
ssdp_flood            424.0  4057.0   4072.0         NaN         NaN   
ssl_renegotiation       NaN  4229.0      NaN         NaN         NaN   
syn_dos                 NaN     NaN      NaN         NaN         NaN   
video_injection         NaN     NaN      NaN         NaN         NaN   

target_fpr             0.0001  
alert_rule        mean of 100  
attack                         
active_wiretap            NaN  
arp_mitm                  NaN  
fuzzing                   NaN  
mirai_botnet             42.0  
os_scan                  30.0  
ssdp_flood             4077.0  
ssl_renegotiation      4035.0  
syn_dos                   NaN  
video_injection           NaN

**False alarm events before the attack start**, for each 100,000 benign packets. An event is a sequence of alerts. Alerts with fewer than 100 packets between them are one event. An operator gets one alarm for one event, so this is the operational measure.

In [7]:
results.pivot(index="attack", columns=["target_fpr", "alert_rule"], values="false_alarm_events_per_100k").round(1)

target_fpr           0.0100                    0.0010                  \
alert_rule        threshold 3 of 5 10 of 20 threshold 3 of 5 10 of 20   
attack                                                                  
active_wiretap        141.0   30.8     25.6     110.3    1.0      1.0   
arp_mitm              221.5   63.6     28.7      96.4    0.5      0.5   
fuzzing               160.5   41.0     13.3      45.6    6.7      1.5   
mirai_botnet          367.2  115.1      5.5      98.7    5.5      0.0   
os_scan                96.3   46.7     19.3      39.6    5.1      1.0   
ssdp_flood            123.5   44.6     24.2      47.7    9.7      4.3   
ssl_renegotiation     134.3   51.0      9.1      58.6    1.5      1.5   
syn_dos               129.5   25.6     15.4      48.7    5.5      2.7   
video_injection       133.6    6.9      1.9      17.2    0.6      0.0   

target_fpr           0.0001                      0.0100      0.0010  \
alert_rule        threshold 3 of 5 10 of 20 mean of 100 mean of 100   
attack                                                                
active_wiretap         28.7    0.0      0.0        31.3        27.2   
arp_mitm               46.2    0.5      0.5        36.4         7.2   
fuzzing                 3.1    0.0      0.0         4.1         0.0   
mirai_botnet           21.9    0.0      0.0        43.9        21.9   
os_scan                 8.1    0.0      0.0        17.2         1.0   
ssdp_flood              1.5    0.0      0.0        19.9         4.8   
ssl_renegotiation      10.6    0.0      0.0         6.6         1.5   
syn_dos                 9.9    1.2      0.2        10.2         2.7   
video_injection         8.1    0.0      0.0         0.3         0.0   

target_fpr             0.0001  
alert_rule        mean of 100  
attack                         
active_wiretap           27.2  
arp_mitm                  0.5  
fuzzing                   0.0  
mirai_botnet             21.9  
os_scan                   0.0  
ssdp_flood                0.0  
ssl_renegotiation         1.0  
syn_dos                   2.2  
video_injection           0.0

**Upper limit (95%) of the false alarm event rate**, for each 100,000 benign packets. Zero events in a limited number of packets does not prove a zero rate. This table shows the largest rate that agrees with the data.

In [8]:
results.pivot(index="attack", columns=["target_fpr", "alert_rule"], values="false_alarm_events_per_100k_upper95").round(1)

target_fpr           0.0100                    0.0010                  \
alert_rule        threshold 3 of 5 10 of 20 threshold 3 of 5 10 of 20   
attack                                                                  
active_wiretap        155.8   38.1     32.5     123.5    3.2      3.2   
arp_mitm              239.9   73.8     35.9     108.8    2.4      2.4   
fuzzing               176.3   49.4     18.5      54.4   10.6      4.0   
mirai_botnet          450.1  165.8     26.0     146.3   26.0     16.4   
os_scan               108.7   55.5     25.3      47.8    8.6      3.2   
ssdp_flood            133.1   50.6     28.7      53.9   12.7      6.5   
ssl_renegotiation     148.7   60.2     13.5      68.4    3.9      3.9   
syn_dos               139.2   30.2     19.0      54.8    7.8      4.5   
video_injection       144.7    9.8      3.7      21.5    2.0      0.9   

target_fpr           0.0001                      0.0100      0.0010  \
alert_rule        threshold 3 of 5 10 of 20 mean of 100 mean of 100   
attack                                                                
active_wiretap         35.9    1.5      1.5        38.7        34.2   
arp_mitm               55.0    2.4      2.4        44.4        11.2   
fuzzing                 6.1    1.5      1.5         7.4         1.5   
mirai_botnet           50.2   16.4     16.4        79.1        50.2   
os_scan                12.3    1.5      1.5        23.0         3.2   
ssdp_flood              3.0    0.8      0.8        24.0         7.1   
ssl_renegotiation      15.3    1.5      1.5        10.4         3.9   
syn_dos                12.9    2.6      1.2        13.2         4.5   
video_injection        11.3    0.9      0.9         1.5         0.9   

target_fpr             0.0001  
alert_rule        mean of 100  
attack                         
active_wiretap           34.2  
arp_mitm                  2.4  
fuzzing                   1.5  
mirai_botnet             50.2  
os_scan                   1.5  
ssdp_flood                0.8  
ssl_renegotiation         3.2  
syn_dos                   3.9  
video_injection           0.9

**Packets with a false alarm before the attack start**, for each 100,000 benign packets. A target rate of 0.001 is equal to 100 for each 100,000. A much larger number shows that the benign traffic changed between the training period and the test period.

In [9]:
results.pivot(index="attack", columns=["target_fpr", "alert_rule"], values="false_alarms_per_100k").round(1)

target_fpr           0.0100                      0.0010                  \
alert_rule        threshold   3 of 5 10 of 20 threshold 3 of 5 10 of 20   
attack                                                                    
active_wiretap      18065.1  17563.1  17806.7     192.8   53.3     53.8   
arp_mitm             4765.1   4359.5   4127.7     154.9   39.0     41.5   
fuzzing               850.3    572.3    297.9     121.5   71.8     27.2   
mirai_botnet         1501.9    526.2    126.1     142.5   16.4      0.0   
os_scan              2522.8   2156.7   1976.2     208.9   64.4     30.4   
ssdp_flood           1342.8   1058.6    810.7     240.8  148.5     89.3   
ssl_renegotiation    1306.0   1082.3    538.4     119.7   44.9     46.5   
syn_dos               902.9    653.7    553.5     216.7  133.5    108.6   
video_injection       269.0     57.6     36.3      25.7    1.3      0.0   

target_fpr           0.0001                      0.0100      0.0010  \
alert_rule        threshold 3 of 5 10 of 20 mean of 100 mean of 100   
attack                                                                
active_wiretap         50.3    0.0      0.0     24767.2     11906.7   
arp_mitm               67.7   16.4     14.9      8090.8       707.2   
fuzzing                 5.6    0.0      0.0       226.7         0.0   
mirai_botnet           21.9    0.0      0.0      2444.6      1606.0   
os_scan                13.2    0.0      0.0      3501.0        74.0   
ssdp_flood              1.8    0.0      0.0      2490.2       413.8   
ssl_renegotiation      11.1    0.0      0.0       801.5       108.1   
syn_dos                26.8   12.4      8.7      1279.5       457.8   
video_injection         9.4    0.0      0.0         0.6         0.0   

target_fpr             0.0001  
alert_rule        mean of 100  
attack                         
active_wiretap         6752.8  
arp_mitm                102.6  
fuzzing                   0.0  
mirai_botnet           1551.2  
os_scan                   0.0  
ssdp_flood                0.0  
ssl_renegotiation        60.6  
syn_dos                 171.0  
video_injection           0.0

**Placebo test.** The same evaluation at 6 false attack starts in benign traffic. The number is the count of false starts that the credibility check accepts. It must be 0. A larger number shows that the result for that setting is not reliable.

In [10]:
results.pivot(index="attack", columns=["target_fpr", "alert_rule"], values="placebo_credible")

target_fpr           0.0100                    0.0010                  \
alert_rule        threshold 3 of 5 10 of 20 threshold 3 of 5 10 of 20   
attack                                                                  
active_wiretap            0      0        0         0      0        0   
arp_mitm                  0      0        0         0      0        0   
fuzzing                   0      0        0         0      1        1   
mirai_botnet              0      0        0         0      0        0   
os_scan                   0      0        0         0      0        0   
ssdp_flood                0      0        0         0      0        0   
ssl_renegotiation         0      0        0         0      0        0   
syn_dos                   0      0        0         0      0        0   
video_injection           0      0        0         0      0        0   

target_fpr           0.0001                      0.0100      0.0010  \
alert_rule        threshold 3 of 5 10 of 20 mean of 100 mean of 100   
attack                                                                
active_wiretap            0      0        0           0           0   
arp_mitm                  0      0        0           2           0   
fuzzing                   1      0        0           0           0   
mirai_botnet              0      0        0           0           0   
os_scan                   0      0        0           0           0   
ssdp_flood                0      0        0           0           0   
ssl_renegotiation         0      0        0           0           0   
syn_dos                   0      0        0           0           0   
video_injection           0      0        0           0           0   

target_fpr             0.0001  
alert_rule        mean of 100  
attack                         
active_wiretap              0  
arp_mitm                    0  
fuzzing                     0  
mirai_botnet                0  
os_scan                     0  
ssdp_flood                  0  
ssl_renegotiation           0  
syn_dos                     0  
video_injection             0

**Packet recall:** the fraction of malicious packets that have an alert.

In [11]:
results.pivot(index="attack", columns=["target_fpr", "alert_rule"], values="packet_recall").round(3)

target_fpr           0.0100                    0.0010                  \
alert_rule        threshold 3 of 5 10 of 20 threshold 3 of 5 10 of 20   
attack                                                                  
active_wiretap        0.216  0.214    0.218     0.003  0.003    0.003   
arp_mitm              0.053  0.052    0.051     0.002  0.002    0.001   
fuzzing               0.021  0.017    0.011     0.019  0.015    0.009   
mirai_botnet          0.884  0.900    0.935     0.882  0.898    0.933   
os_scan               1.000  0.975    0.960     1.000  0.975    0.959   
ssdp_flood            1.000  1.000    1.000     1.000  1.000    1.000   
ssl_renegotiation     0.946  0.730    0.240     0.889  0.676    0.205   
syn_dos               0.251  0.206    0.140     0.213  0.177    0.122   
video_injection       0.098  0.076    0.049     0.003  0.002    0.001   

target_fpr           0.0001                      0.0100      0.0010  \
alert_rule        threshold 3 of 5 10 of 20 mean of 100 mean of 100   
attack                                                                
active_wiretap        0.001  0.001    0.001       0.289       0.085   
arp_mitm              0.001  0.001    0.001       0.095       0.023   
fuzzing               0.004  0.000    0.000       0.011       0.000   
mirai_botnet          0.858  0.878    0.912       1.000       1.000   
os_scan               1.000  0.975    0.959       1.000       1.000   
ssdp_flood            1.000  0.999    1.000       1.000       1.000   
ssl_renegotiation     0.853  0.657    0.192       0.902       0.877   
syn_dos               0.199  0.164    0.108       0.204       0.171   
video_injection       0.000  0.000    0.000       0.027       0.001   

target_fpr             0.0001  
alert_rule        mean of 100  
attack                         
active_wiretap          0.054  
arp_mitm                0.004  
fuzzing                 0.000  
mirai_botnet            1.000  
os_scan                 0.999  
ssdp_flood              1.000  
ssl_renegotiation       0.877  
syn_dos                 0.145  
video_injection         0.000

## 3. Keep the output

Select **Save Version > Save & Run All (Commit)**. The folder `baseline` then has `results.csv`, the test scores for each capture, the trained models (`models/`), and packets near each attack start for the dashboard (`demo/`). The file `mlflow.db` has the MLflow records.

To send the results, run this cell and copy the output:

In [12]:
print(results.to_csv(index=False))

attack,target_fpr,alert_rule,smooth_window,threshold,k,n,detected,delay_packets,benign_packets_before_attack,false_alarms_before_attack,false_alarms_per_100k,false_alarm_events,false_alarm_events_per_100k,false_alarm_events_per_100k_upper95,packet_recall,alert_lift,detection_credible,chance_delay_probability,delay_credible,placebo_credible,placebo_starts
arp_mitm,0.01,threshold,1,0.17439010244252667,1,1,True,0.0,195000,9292,4765.128205128205,432,221.53846153846155,239.8903601235209,0.05311925900572091,1.1147498392289688,False,0.047651282051281996,False,0,6
arp_mitm,0.01,3 of 5,1,0.17439010244252667,3,5,True,0.0,195000,8501,4359.48717948718,124,63.58974358974359,73.815769364572,0.052102033403418575,1.1951413379210236,False,0.04359487179487176,False,0,6
arp_mitm,0.01,10 of 20,1,0.17439010244252667,10,20,True,0.0,195000,8049,4127.692307692308,56,28.717948717948715,35.87712151242455,0.05059584098799237,1.2257658085052194,False,0.0412769230769231,False,0,6
arp_mitm,0.001,threshold,1,0.80294